In [2]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
criterion = nn.MSELoss()

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

for epoch in range(1000):
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("PyTorch XOR Results:")
with torch.no_grad():
    for i in range(4):
        pred = model(X[i])
        print(f"  {X[i].tolist()} -> {pred.item():.4f} (expected {y[i].item()})")

PyTorch XOR Results:
  [0.0, 0.0] -> 0.4539 (expected 0.0)
  [0.0, 1.0] -> 0.6674 (expected 1.0)
  [1.0, 0.0] -> 0.4233 (expected 1.0)
  [1.0, 1.0] -> 0.4313 (expected 0.0)


In [2]:
x = torch.zeros(3, 4)

print(x.ndim)   # 2
print(x.shape)  # torch.Size([3, 4])

2
torch.Size([3, 4])


In [4]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [4.0, 5.0, 6.0],
])

print(x.shape)  # torch.Size([2, 3])
print(x.ndim)   # 2

torch.Size([3, 3])
2


In [2]:
import torch
x = torch.randn(3, requires_grad=True)
print(x)
y = x ** 2 + 3 * x
z = y.sum()
z.backward()
print(x.grad)  # dz/dx = 2x + 3

tensor([-1.0921, -0.3564,  0.6221], requires_grad=True)
tensor([0.8157, 2.2871, 4.2441])


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
import numpy as np


def init_embeddings(vocab_size, dim, seed=0):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(vocab_size, dim))
    W_prime = rng.normal(0, 0.1, size=(vocab_size, dim))
    return W, W_prime

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


word_to_id = {
    "the": 0,
    "cat": 1,
    "sat": 2,
    "on": 3,
    "mat": 4,
}

W, W_prime = init_embeddings(
    vocab_size=len(word_to_id),
    dim=3,
    seed=0,
)

center_id = word_to_id["cat"]
context_id = word_to_id["sat"]

center_vector = W[center_id]
context_vector = W_prime[context_id]

score = center_vector @ context_vector
probability = sigmoid(score)

print("cat 中心词向量:", center_vector)
print("sat 上下文向量:", context_vector)
print("点积:", score)
print("真实词对概率:", probability)

cat 中心词向量: [ 0.01049001 -0.05356694  0.03615951]
sat 上下文向量: [ 0.13664635 -0.06651947  0.03515101]
点积: 0.006267708950192178
真实词对概率: 0.5015669221079478


In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

src = "The cats are running."
inputs = tok(src, return_tensors="pt")

out = model.generate(
    **inputs,
    forced_bos_token_id=tok.convert_tokens_to_ids("fra_Latn"),
    num_beams=5,
    length_penalty=1.0,
    max_new_tokens=64,
)
print(tok.batch_decode(out, skip_special_tokens=True)[0])

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

C:\Users\14300\Documents\GitHub\ai-engineering-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\14300\.cache\huggingface\hub\models--facebook--nllb-200-distilled-600M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Les chats courent.
